In [ ]:
import tensorflow as tf
import pandas as pd
import numpy as np
from pathlib import Path
import os
from sklearn.model_selection import train_test_split

gpus = tf.config.experimental.list_physical_devices('GPU')
if gpus:
    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
    except RuntimeError:
        pass

SIMILAR_CLASS_GROUPS = {
    'aloo_group': ['aloo_matar', 'aloo_methi', 'aloo_tikki'],
    'chicken_group': ['chicken_tikka', 'chicken_tikka_masala', 'chicken_razala'],
    'dal_group': ['dal_makhani', 'dal_tadka'],
    'paneer_group': ['kadai_paneer', 'palak_paneer'],
    'sweet_milk_group': ['double_ka_meetha', 'qubani_ka_meetha'],
    'sheer_group': ['sheer_korma', 'sheera'],
    'dessert_group': ['chak_hao_kheer', 'chhena_kheeri']
}

image_dir = Path('/kaggle/input/indian-food-images-dataset/Indian Food Images/Indian Food Images')
filepath = list(image_dir.glob(r'**/*.jpg'))
label = list(map(lambda x: os.path.split(os.path.split(x)[0])[1], filepath))
filepath = pd.Series(filepath, name='Filepath').astype(str)
label = pd.Series(label, name='Label')
image_df = pd.concat([filepath, label], axis=1).sample(frac=1.0, random_state=1).reset_index(drop=True)
train_df, test_df = train_test_split(image_df, test_size=0.30, shuffle=True, random_state=1)


In [ ]:
import tensorflow as tf

def preprocess_input(x):
    return tf.keras.applications.efficientnet.preprocess_input(x)

img_size = (380, 380)
batch_size = 16

train_datagen = tf.keras.preprocessing.image.ImageDataGenerator(
    preprocessing_function=preprocess_input,
    validation_split=0.2,
    rotation_range=30,
    width_shift_range=0.3,
    height_shift_range=0.3,
    horizontal_flip=True,
    vertical_flip=True,
    zoom_range=0.4,
    shear_range=0.3,
    brightness_range=[0.6, 1.4],
    channel_shift_range=50.0,
    fill_mode='nearest'
)

test_datagen = tf.keras.preprocessing.image.ImageDataGenerator(
    preprocessing_function=preprocess_input
)

train_generator = train_datagen.flow_from_dataframe(
    dataframe=train_df,
    x_col='Filepath',
    y_col='Label',
    target_size=img_size,
    batch_size=batch_size,
    color_mode='rgb',
    class_mode='categorical',
    shuffle=True,
    seed=42,
    subset='training'
)

val_generator = train_datagen.flow_from_dataframe(
    dataframe=train_df,
    x_col='Filepath',
    y_col='Label',
    target_size=img_size,
    batch_size=batch_size,
    color_mode='rgb',
    class_mode='categorical',
    shuffle=True,
    seed=42,
    subset='validation'
)

test_generator = test_datagen.flow_from_dataframe(
    dataframe=test_df,
    x_col='Filepath',
    y_col='Label',
    target_size=img_size,
    batch_size=batch_size,
    color_mode='rgb',
    class_mode='categorical',
    shuffle=False
)

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers

def create_advanced_model(input_shape, num_classes):
    base_model = tf.keras.applications.EfficientNetB3(
        include_top=False,
        weights='imagenet',
        input_shape=input_shape,
        pooling=None
    )
    base_model.trainable = False
    inputs = tf.keras.Input(shape=input_shape)
    x = base_model(inputs, training=False)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(0.6)(x)
    x = layers.Dense(1024, activation='swish', kernel_regularizer=tf.keras.regularizers.l2(0.001))(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.5)(x)
    x = layers.Dense(512, activation='swish', kernel_regularizer=tf.keras.regularizers.l2(0.001))(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.4)(x)
    x = layers.Dense(256, activation='swish', kernel_regularizer=tf.keras.regularizers.l2(0.001))(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.3)(x)
    outputs = layers.Dense(num_classes, activation='softmax')(x)
    model = tf.keras.Model(inputs, outputs)
    return model, base_model

num_classes = 80
model, base_model = create_advanced_model((img_size[0], img_size[1], 3), num_classes)

optimizer = tf.keras.optimizers.Adam(learning_rate=0.0005)
model.compile(
    optimizer=optimizer,
    loss='categorical_crossentropy',
    metrics=[
        'accuracy',
        tf.keras.metrics.TopKCategoricalAccuracy(k=3, name='top_3_accuracy'),
        tf.keras.metrics.TopKCategoricalAccuracy(k=5, name='top_5_accuracy')
    ]
)

callbacks_phase1 = [
    tf.keras.callbacks.EarlyStopping(monitor='val_accuracy', patience=12, restore_best_weights=True, mode='max', verbose=0),
    tf.keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=6, min_lr=1e-7, verbose=0),
    tf.keras.callbacks.ModelCheckpoint('best_model_phase1.keras', monitor='val_accuracy', save_best_only=True, mode='max', verbose=0)
]

history_phase1 = model.fit(
    train_generator,
    epochs=40,
    validation_data=val_generator,
    callbacks=callbacks_phase1,
    verbose=0
)

In [ ]:
import tensorflow as tf
import numpy as np

class SimilarClassAwareLoss:
    def __init__(self, similar_groups, num_classes, class_names, alpha=0.3):
        self.similar_groups = similar_groups
        self.num_classes = num_classes
        self.class_names = class_names
        self.alpha = alpha
        self.similarity_matrix = self._build_similarity_matrix()
    def _build_similarity_matrix(self):
        matrix = np.eye(self.num_classes)
        for _, classes in self.similar_groups.items():
            indices = [list(self.class_names).index(cls) for cls in classes if cls in self.class_names]
            for i in indices:
                for j in indices:
                    if i != j:
                        matrix[i, j] = self.alpha
        return tf.constant(matrix, dtype=tf.float32)
    def __call__(self, y_true, y_pred):
        base_loss = tf.keras.losses.categorical_crossentropy(y_true, y_pred)
        max_probs = tf.reduce_max(y_pred, axis=-1)
        confidence_penalty = tf.reduce_mean(1.0 - max_probs)
        return base_loss + confidence_penalty * self.alpha

class_names = list(train_generator.class_indices.keys())
custom_loss = SimilarClassAwareLoss(SIMILAR_CLASS_GROUPS, num_classes, class_names, alpha=0.3)

In [ ]:
import tensorflow as tf
import numpy as np

def predict_with_tta(model, test_generator, n_augment=5):
    all_images = []
    all_labels = []
    test_generator.reset()
    for i in range(len(test_generator)):
        batch_images, batch_labels = test_generator[i]
        all_images.append(batch_images)
        all_labels.append(batch_labels)
    all_images = np.vstack(all_images)
    all_labels = np.vstack(all_labels)
    all_predictions = []
    for aug_idx in range(n_augment):
        aug_images = all_images.copy()
        if aug_idx == 1:
            aug_images = tf.image.flip_left_right(aug_images).numpy()
        elif aug_idx == 2:
            aug_images = tf.image.random_brightness(aug_images, 0.2).numpy()
        elif aug_idx == 3:
            aug_images = tf.image.random_contrast(aug_images, 0.8, 1.2).numpy()
        elif aug_idx == 4:
            aug_images = tf.keras.preprocessing.image.random_rotation(aug_images, 15, row_axis=0, col_axis=1, channel_axis=3)
        batch_predictions = model.predict(aug_images, verbose=0, batch_size=32)
        all_predictions.append(batch_predictions)
    final_predictions = np.mean(all_predictions, axis=0)
    return final_predictions

def simple_ensemble(models_list, test_generator):
    all_predictions = []
    for model_path in models_list:
        try:
            m = tf.keras.models.load_model(model_path)
            predictions = m.predict(test_generator, verbose=0)
            all_predictions.append(predictions)
        except Exception:
            pass
    if len(all_predictions) > 1:
        return np.mean(all_predictions, axis=0)
    return all_predictions[0] if all_predictions else None

def calibrate_predictions(predictions, temperature=1.5):
    calibrated = predictions ** (1/temperature)
    calibrated = calibrated / np.sum(calibrated, axis=1, keepdims=True)
    return calibrated

model_paths = ['final_model.keras', 'best_model_phase1.keras']
ensemble_predictions = simple_ensemble(model_paths, test_generator)
if ensemble_predictions is None:
    ensemble_predictions = model.predict(test_generator, verbose=0)
ensemble_accuracy = np.mean(np.argmax(ensemble_predictions, axis=1) == test_generator.classes)
calibrated_predictions = calibrate_predictions(ensemble_predictions, temperature=1.3)
calibrated_accuracy = np.mean(np.argmax(calibrated_predictions, axis=1) == test_generator.classes)
tta_predictions = None
if ensemble_accuracy > 0.64:
    tta_predictions = predict_with_tta(model, test_generator, n_augment=3)
    tta_accuracy = np.mean(np.argmax(tta_predictions, axis=1) == test_generator.classes)
combined_pred = None
if tta_predictions is not None:
    combined_pred = (ensemble_predictions + tta_predictions) / 2
    combined_accuracy = np.mean(np.argmax(combined_pred, axis=1) == test_generator.classes)

In [ ]:
import numpy as np
import tensorflow as tf

class_names = list(test_generator.class_indices.keys())
y_true = test_generator.classes
y_pred_base = np.argmax(ensemble_predictions, axis=1)
class_accuracy = {}
for i, class_name in enumerate(class_names):
    mask = (y_true == i)
    if np.sum(mask) > 0:
        acc = np.mean(y_pred_base[mask] == i)
        class_accuracy[class_name] = acc
worst_classes = sorted(class_accuracy.items(), key=lambda x: x[1])[:10]

def create_class_weights(class_accuracy_dict, boost_factor=2.0):
    weights = {}
    for i, (cls, acc) in enumerate(class_accuracy_dict.items()):
        weight = 1.0 + (1.0 - acc) * boost_factor
        weights[i] = weight
    return weights

class_weights = create_class_weights(dict(worst_classes), boost_factor=3.0)
base_model = model.layers[1]
for layer in base_model.layers[-30:]:
    layer.trainable = True
model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.00001), loss='categorical_crossentropy', metrics=['accuracy'])
history = model.fit(train_generator, epochs=3, validation_data=val_generator, class_weight=class_weights, verbose=0)
improved_results = model.evaluate(test_generator, verbose=0)
model.save('improved_final_model.keras')

In [ ]:
import tensorflow as tf
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix, roc_curve, auc, precision_recall_curve, average_precision_score
from sklearn.preprocessing import label_binarize
from itertools import cycle
import warnings

warnings.filterwarnings('ignore')

try:
    model = tf.keras.models.load_model('improved_final_model.keras')
    model.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy', tf.keras.metrics.TopKCategoricalAccuracy(k=3, name='top_3_accuracy'), tf.keras.metrics.TopKCategororicalAccuracy(k=5, name='top_5_accuracy')])
except Exception:
    try:
        model = tf.keras.models.load_model('final_model.keras')
        model.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy', tf.keras.metrics.TopKCategoricalAccuracy(k=3, name='top_3_accuracy'), tf.keras.metrics.TopKCategoricalAccuracy(k=5, name='top_5_accuracy')])
    except Exception:
        pass

assert 'test_generator' in locals()

class_names = list(test_generator.class_indices.keys())
num_classes = len(class_names)
test_generator.reset()
y_pred_probs = model.predict(test_generator, verbose=0)
y_pred = np.argmax(y_pred_probs, axis=1)
y_true = test_generator.classes
y_true_binarized = label_binarize(y_true, classes=range(num_classes))
test_results = model.evaluate(test_generator, verbose=0)

top_k_metrics = {'Top-1 Accuracy': test_results[1], 'Top-3 Accuracy': test_results[2], 'Top-5 Accuracy': test_results[3]}
plt.figure(figsize=(10, 6))
sns.barplot(x=list(top_k_metrics.keys()), y=list(top_k_metrics.values()))
plt.title('Top-K Accuracy Comparison')
plt.ylabel('Accuracy')
plt.ylim(0, max(test_results[3] * 1.2, 1.0))
for i, (k, v) in enumerate(top_k_metrics.items()):
    plt.text(i, v + 0.02, f"{v:.2%}", ha='center')
plt.show()

if 'image_df' in locals():
    plt.figure(figsize=(14, 20))
    sns.countplot(y=image_df['Label'], order=image_df['Label'].value_counts().index)
    plt.title('Dataset Class Distribution')
    plt.xlabel('Number of Images')
    plt.ylabel('Food Class')
    plt.tick_params(axis='y', labelsize=8)
    plt.tight_layout()
    plt.show()

def plot_training_history(history, title):
    if not (history and hasattr(history, 'history') and 'accuracy' in history.history):
        return
    acc = history.history['accuracy']
    val_acc = history.history['val_accuracy']
    loss = history.history['loss']
    val_loss = history.history['val_loss']
    epochs = range(1, len(acc) + 1)
    plt.figure(figsize=(15, 6))
    plt.subplot(1, 2, 1)
    plt.plot(epochs, acc, 'bo-', label='Training Accuracy')
    plt.plot(epochs, val_acc, 'ro-', label='Validation Accuracy')
    plt.title(f'Accuracy - {title}')
    plt.xlabel('Epochs')
    plt.ylabel('Accuracy')
    plt.legend()
    plt.grid(True)
    plt.subplot(1, 2, 2)
    plt.plot(epochs, loss, 'bo-', label='Training Loss')
    plt.plot(epochs, val_loss, 'ro-', label='Validation Loss')
    plt.title(f'Loss - {title}')
    plt.xlabel('Epochs')
    plt.ylabel('Loss')
    plt.legend()
    plt.grid(True)
    plt.suptitle(f'Training & Validation Metrics: {title}')
    plt.tight_layout(rect=[0, 0.03, 1, 0.95])
    plt.show()

if 'history_phase1' in locals():
    plot_training_history(history_phase1, "Phase 1: Head Training")
if 'history_phase2' in locals():
    plot_training_history(history_phase2, "Phase 2: Fine-Tuning")
if 'history' in locals() and 'history_phase1' in locals() and history is not history_phase1 and ('history_phase2' not in locals() or history is not history_phase2):
    plot_training_history(history, "Phase 3: Weak Class Fine-Tuning")

report = classification_report(y_true, y_pred, target_names=class_names)
report_dict = classification_report(y_true, y_pred, target_names=class_names, output_dict=True)
class_performance = {}
for class_name, metrics in report_dict.items():
    if class_name in class_names:
        class_performance[class_name] = metrics['f1-score']
sorted_performance = sorted(class_performance.items(), key=lambda x: x[1])
worst_10 = sorted_performance[:10]
best_10 = sorted_performance[-10:][::-1]
plt.figure(figsize=(20, 8))
plt.subplot(1, 2, 1)
sns.barplot(x=[val for _, val in worst_10], y=[key for key, _ in worst_10])
plt.title('Top 10 Worst Performing Classes (F1)')
plt.xlabel('F1-Score')
plt.ylabel('Food Class')
plt.xlim(0, 1.0)
plt.subplot(1, 2, 2)
sns.barplot(x=[val for _, val in best_10], y=[key for key, _ in best_10])
plt.title('Top 10 Best Performing Classes (F1)')
plt.xlabel('F1-Score')
plt.ylabel('')
plt.xlim(0, 1.0)
plt.tight_layout()
plt.show()

cm = confusion_matrix(y_true, y_pred)
cm_normalized = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]
cm_normalized = np.nan_to_num(cm_normalized)
plt.figure(figsize=(28, 28))
sns.heatmap(cm_normalized, annot=False, fmt=".2f", cmap='Blues', xticklabels=class_names, yticklabels=class_names)
plt.title('Normalized Confusion Matrix')
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.xticks(rotation=90, fontsize=8)
plt.yticks(rotation=0, fontsize=8)
plt.tight_layout()
plt.show()

cm_errors = cm.copy()
np.fill_diagonal(cm_errors, 0)
top_errors = []
for _ in range(10):
    max_idx = np.argmax(cm_errors)
    true_idx, pred_idx = np.unravel_index(max_idx, cm_errors.shape)
    error_count = cm_errors[true_idx, pred_idx]
    if error_count == 0:
        break
    top_errors.append((class_names[true_idx], class_names[pred_idx], error_count))
    cm_errors[true_idx, pred_idx] = 0
plt.figure(figsize=(12, 8))
error_labels = [f"{t} -> {p}" for t, p, _ in top_errors]
error_counts = [c for _, _, c in top_errors]
sns.barplot(x=error_counts, y=error_labels)
plt.title('Top 10 Most Common Misclassifications')
plt.xlabel('Count')
plt.show()

fpr = {}
tpr = {}
roc_auc = {}
for i in range(num_classes):
    fpr[i], tpr[i], _ = roc_curve(y_true_binarized[:, i], y_pred_probs[:, i])
    roc_auc[i] = auc(fpr[i], tpr[i])
fpr["micro"], tpr["micro"], _ = roc_curve(y_true_binarized.ravel(), y_pred_probs.ravel())
roc_auc["micro"] = auc(fpr["micro"], tpr["micro"])
all_fpr = np.unique(np.concatenate([fpr[i] for i in range(num_classes)]))
mean_tpr = np.zeros_like(all_fpr)
for i in range(num_classes):
    mean_tpr += np.interp(all_fpr, fpr[i], tpr[i])
mean_tpr /= num_classes
fpr["macro"] = all_fpr
tpr["macro"] = mean_tpr
roc_auc["macro"] = auc(fpr["macro"], tpr["macro"])
plt.figure(figsize=(12, 9))
plt.plot(fpr["micro"], tpr["micro"], label=f'Micro-average ROC (AUC = {roc_auc["micro"]:.3f})', linestyle=':', linewidth=4)
plt.plot(fpr["macro"], tpr["macro"], label=f'Macro-average ROC (AUC = {roc_auc["macro"]:.3f})', linestyle=':', linewidth=4)
plt.plot([0, 1], [0, 1], 'k--', lw=2, label='Chance')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Multi-class ROC Curves')
plt.legend(loc="lower right")
plt.grid(True)
plt.show()

precision, recall, _ = precision_recall_curve(y_true_binarized.ravel(), y_pred_probs.ravel())
average_precision = average_precision_score(y_true_binarized, y_pred_probs, average="micro")
plt.figure(figsize=(12, 9))
plt.step(recall, precision, where='post', label=f'Micro-average PR (AUPR = {average_precision:.3f})')
plt.xlabel('Recall')
plt.ylabel('Precision')
plt.ylim([0.0, 1.05])
plt.xlim([0.0, 1.0])
plt.title('Micro-Average Multi-class Precision-Recall')
plt.legend(loc="upper right")
plt.grid(True)
plt.show()

test_generator.reset()
images, true_labels_one_hot = next(test_generator)
true_labels_idx = np.argmax(true_labels_one_hot, axis=1)
pred_probs_batch = model.predict(images, verbose=0)
pred_labels_idx = np.argmax(pred_probs_batch, axis=1)
plt.figure(figsize=(20, 22))
num_images = min(25, len(images))
for i in range(num_images):
    plt.subplot(5, 5, i + 1)
    img = images[i]
    img = (img + 1) / 2.0
    img = np.clip(img, 0, 1)
    plt.imshow(img)
    plt.axis('off')
    true_name = class_names[true_labels_idx[i]]
    pred_name = class_names[pred_labels_idx[i]]
    pred_confidence = pred_probs_batch[i][pred_labels_idx[i]]
    if true_name == pred_name:
        title = f"True: {true_name}\nPred: {pred_name}\nConf: {pred_confidence:.2f}"
    else:
        top_2_idx = np.argsort(pred_probs_batch[i])[-2:]
        pred_2_name = class_names[top_2_idx[-2]]
        pred_2_conf = pred_probs_batch[i][top_2_idx[-2]]
        title = f"True: {true_name}\nPred 1: {pred_name} ({pred_confidence:.2f})\nPred 2: {pred_2_name} ({pred_2_conf:.2f})"
    plt.title(title, fontsize=9)
plt.suptitle('Example Model Predictions')
plt.tight_layout(rect=[0, 0.03, 1, 0.96])
plt.show()